In [1]:
import pandas as pd

In [2]:
exchange_rate = pd.read_csv('exchange_rate.csv')
remit = pd.read_csv('worker_ remittances.csv')

In [3]:
exchange_rate=exchange_rate.drop(columns = ['Dataset','Series Key','Series','Observation Status Comment','Unit','Observation Status'])
exchange_rate = exchange_rate.rename(columns={'Observation Date':'date','Observation Value':'pkr_exchange_rate'})
exchange_rate["date"] = exchange_rate["date"].astype("datetime64[s]")

In [4]:
remit = remit.rename(columns={'Date':'date','Workers Remittances (Million USD)':'remittance_usd(millions)'})
remit['date'] = pd.to_datetime(remit['date'], format='%b-%Y') + pd.offsets.MonthEnd(0)
remit_exchange_rate_merged = pd.merge(remit , exchange_rate,how='left', on='date')

In [5]:
remit_exchange_rate_merged['usd_exchange_rate'] = 1/ remit_exchange_rate_merged['pkr_exchange_rate']
remit_exchange_rate_merged['remittance_pkr(100_millions)'] = (remit_exchange_rate_merged['remittance_usd(millions)'] * remit_exchange_rate_merged['usd_exchange_rate'])/100
remit_exchange_rate_merged['remittance_pct_change'] = remit_exchange_rate_merged['remittance_usd(millions)'].pct_change()*100
remit_exchange_rate_merged['pkr_pct_change'] = remit_exchange_rate_merged['usd_exchange_rate']
remit_exchange_rate_merged['roling_corr_remittance_pkr_pct_change'] = remit_exchange_rate_merged['remittance_pct_change'].rolling(12).corr(remit_exchange_rate_merged['pkr_pct_change'])
remit_exchange_rate_merged['remittance_year_by_year'] = remit_exchange_rate_merged['remittance_usd(millions)'].pct_change(12)*100
remit_exchange_rate_merged['remittance_3mo_avg'] = remit_exchange_rate_merged['remittance_usd(millions)'].rolling(3).mean()
remit_exchange_rate_merged['remittance_12mo_avg'] = remit_exchange_rate_merged['remittance_usd(millions)'].rolling(12).mean()
remit_exchange_rate_merged['fiscal_year'] = remit_exchange_rate_merged['date'].dt.year.where(remit_exchange_rate_merged['date'].dt.month<7,remit_exchange_rate_merged['date'].dt.year +1) 

In [6]:
remit_exchange_rate_merged.to_csv('remittance and exchange rate data.csv')